In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cse250a.modules.config import config
from cse250a.modules.core import TIME_CARDS, Gas, Latent, Network, Sensor, Time, Weather
from cse250a.modules.data import Discretizer, blocked_split, load_air_quality
from cse250a.modules.models import DiscreteBN, HourOnlyBaseline

SPLIT_SEEDS = (0, 1, 2)
RESTART_SEEDS = (0, 1, 2, 3, 4)

# 1. Set-up


In [ ]:
df = load_air_quality( config.paths.data_dir / "air_quality" / "AirQualityUCI.csv") 

GASES = (Gas.CO, Gas.NOX, Gas.NO2, Gas.NMHC)
WEATHER = (Weather.T, Weather.AH)
SENSORS = tuple(Sensor)

before = len(df)
df = df.dropna(subset=[Time.HOUR, *WEATHER])
print(f"dropped {before - len(df)} rows with missing weather; {len(df)} left")

splits = {}
for seed in SPLIT_SEEDS:
    train_raw, val_raw, _ = blocked_split(df, seed=seed)  
    disc = Discretizer(n_bins=config.bins.n_bins).fit(
        train_raw, [*SENSORS, *GASES, *WEATHER]
    )
    splits[seed] = {
        "train_raw": train_raw,
        "train": disc.transform(train_raw),
        "val": disc.transform(val_raw),
        "cards": {**disc.cards, **TIME_CARDS},
    }
pd.DataFrame(
    {s: {"train": len(v["train"]), "val": len(v["val"])} for s, v in splits.items()}
).rename_axis("seed", axis=1)

# 2. Network 